# Student Information

- **Name:** Siddharth Tripathi
- **Email ID:** sidmsi532004@gmail.com
- **Enrollment No.:** 473611


# Titanic Dynamics Submission

Use this notebook to build and vote Kaggle submission candidates.
The comparison notebook stays separate. This notebook is for submission modeling only.

Current top candidates:

- `woman_child_master_rule`
- `merge_woman_child_master_rule_plus_woman_child_age_rule`
- `woman_child_age_rule`

Death-side candidate:

- `dynamics_death_rule`

The notebook votes across the candidate set after scoring them.

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
uploads = root / "kaggle_uploads"
uploads.mkdir(exist_ok=True)

train_url = "https://raw.githubusercontent.com/sid0sid-ops/titanic-data-to-discovery/main/kaggle/train.csv"
test_url = "https://raw.githubusercontent.com/sid0sid-ops/titanic-data-to-discovery/main/kaggle/test.csv"

if not (uploads / "train.csv").exists():
    pd.read_csv(train_url).to_csv(uploads / "train.csv", index=False)
if not (uploads / "test.csv").exists():
    pd.read_csv(test_url).to_csv(uploads / "test.csv", index=False)

train = pd.read_csv(uploads / "train.csv")
test = pd.read_csv(uploads / "test.csv")
print("train:", train.shape, "test:", test.shape)


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

OUT_PATH = uploads / "submission.csv"

def fail(message: str) -> None:
    raise SystemExit(f"ERROR: {message}")

def prepare(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame.columns = frame.columns.str.strip().str.lower()
    frame["title"] = frame["name"].astype(str).str.extract(r",\s*([^.]+)\.", expand=False)
    frame["title"] = frame["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}).fillna("Rare")
    frame["embarked"] = frame["embarked"].fillna("U")
    frame["surname"] = frame["name"].astype(str).str.split(",", n=1).str[0].str.strip()
    return frame

def fill_age(train: pd.DataFrame, frame: pd.DataFrame) -> pd.Series:
    train = prepare(train)
    frame = prepare(frame)
    medians = train.groupby(["pclass", "sex"], dropna=False)["age"].median()
    global_median = train["age"].median()
    age = frame["age"].copy()
    missing = age.isna()
    if missing.any():
        keys = list(zip(frame.loc[missing, "pclass"], frame.loc[missing, "sex"]))
        age.loc[missing] = [medians.get(key, global_median) for key in keys]
    return age

def engineered_frame(train: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    frame = prepare(frame)
    frame = frame.copy()
    frame["family_size"] = frame["sibsp"] + frame["parch"] + 1
    frame["is_alone"] = (frame["family_size"] == 1).astype(int)
    frame["cabin_known"] = frame["cabin"].notna().astype(int)
    frame["deck"] = frame["cabin"].astype(str).str[0].replace("n", "U").fillna("U")
    frame["ticket_prefix"] = (frame["ticket"].astype(str).str.replace(r"\d+", "", regex=True).str.replace(r"\s+", "", regex=True).str.replace(r"[./-]+", "", regex=True).str.strip().replace("", "NUM"))
    frame["fare_per_person"] = frame["fare"] / frame["family_size"].clip(lower=1)
    frame["age_filled"] = fill_age(train, frame)
    frame["pclass"] = frame["pclass"].astype(str)
    frame["embarked"] = frame["embarked"].fillna("U").astype(str)
    frame["title"] = frame["title"].fillna("Rare").astype(str)
    return frame[["pclass", "sex", "embarked", "title", "deck", "ticket_prefix", "family_size", "is_alone", "cabin_known", "fare_per_person", "age_filled"]]

def build_model() -> Pipeline:
    num_cols = ["family_size", "is_alone", "cabin_known", "fare_per_person", "age_filled"]
    cat_cols = ["pclass", "sex", "embarked", "title", "deck", "ticket_prefix"]
    pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols), ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)], remainder="drop")
    return Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=2000, solver="liblinear", C=1.0, random_state=42))])

def predict_ml(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    X_train = engineered_frame(train, train)
    X_test = engineered_frame(train, test)
    model = build_model()
    model.fit(X_train, train["Survived"].astype(int))
    pred = model.predict(X_test).astype(int)
    return pd.DataFrame({"PassengerId": test["PassengerId"].astype(int), "Survived": pred})

def predict_group_table(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    combined = pd.concat([train.assign(_is_train=True), test.assign(_is_train=False)], ignore_index=True)
    combined = prepare(combined)
    combined["age_filled"] = fill_age(combined.loc[combined["_is_train"]].copy(), combined)
    combined["fare_band"] = pd.qcut(combined["fare"].fillna(combined["fare"].median()), q=4, duplicates="drop").astype(str)
    keys = ["sex", "pclass", "embarked", "title"]
    stats = combined.loc[combined["_is_train"]].groupby(keys, dropna=False)["survived"].agg(["mean", "count"]).reset_index()
    low = stats[(stats["count"] >= 20) & (stats["mean"] <= 0.15)]
    high = stats[(stats["count"] >= 20) & (stats["mean"] >= 0.85)]
    low_keys = {tuple(row[k] for k in keys) for _, row in low.iterrows()}
    high_keys = {tuple(row[k] for k in keys) for _, row in high.iterrows()}
    pred = ((combined["sex"] == "female") | (combined["title"] == "Master") | (combined["age_filled"] <= 14)).astype(int)
    row_keys = list(zip(*(combined[k] for k in keys)))
    for idx, key in enumerate(row_keys):
        if key in low_keys:
            pred.iloc[idx] = 0
        elif key in high_keys:
            pred.iloc[idx] = 1
    test_rows = ~combined["_is_train"]
    return pd.DataFrame({"PassengerId": combined.loc[test_rows, "passengerid"].astype(int), "Survived": pred.loc[test_rows].astype(int)})

def predict_male_third_class(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    combined = pd.concat([train.assign(_is_train=True), test.assign(_is_train=False)], ignore_index=True)
    combined = prepare(combined)
    pred = (combined["sex"] == "female").astype(int)
    pred[(combined["sex"] == "male") & (combined["pclass"] == 3)] = 0
    pred[(combined["sex"] == "male") & (combined["pclass"] == 1) & (combined["title"] == "Master")] = 1
    pred[(combined["sex"] == "female") & (combined["pclass"] <= 2)] = 1
    test_rows = ~combined["_is_train"]
    return pd.DataFrame({"PassengerId": combined.loc[test_rows, "passengerid"].astype(int), "Survived": pred.loc[test_rows].astype(int)})

def predict_death_rule(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    combined = pd.concat([train.assign(_is_train=True), test.assign(_is_train=False)], ignore_index=True)
    combined = prepare(combined)
    combined["age_filled"] = fill_age(combined.loc[combined["_is_train"]].copy(), combined)
    pred = (combined["sex"] == "female").astype(int)
    death = (combined["sex"] == "male") & (((combined["pclass"] == 2) & (combined["title"] == "Rev")) | ((combined["pclass"] == 3) & (combined["title"] == "Mr")) | ((combined["pclass"] == 3) & (combined["embarked"] == "S") & (combined["title"] == "Mr")))
    pred[death] = 0
    pred[(combined["sex"] == "male") & (combined["pclass"] == 1) & (combined["title"] == "Master")] = 1
    pred[(combined["age_filled"] <= 14) & (combined["sex"] == "female")] = 1
    test_rows = ~combined["_is_train"]
    return pd.DataFrame({"PassengerId": combined.loc[test_rows, "passengerid"].astype(int), "Survived": pred.loc[test_rows].astype(int)})

def gravity_rules(frame: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    return [("female", frame["sex"] == "female"), ("child", frame["age_filled"] <= 14), ("female_p1", (frame["sex"] == "female") & (frame["pclass"] == 1)), ("female_p2", (frame["sex"] == "female") & (frame["pclass"] == 2)), ("male_p3", (frame["sex"] == "male") & (frame["pclass"] == 3)), ("male_p3_mr", (frame["sex"] == "male") & (frame["pclass"] == 3) & (frame["title"] == "Mr")), ("male_p2_rev", (frame["sex"] == "male") & (frame["pclass"] == 2) & (frame["title"] == "Rev")), ("male_p3_s", (frame["sex"] == "male") & (frame["pclass"] == 3) & (frame["embarked"] == "S")), ("male_p3_q_master", (frame["sex"] == "male") & (frame["pclass"] == 3) & (frame["embarked"] == "Q") & (frame["title"] == "Master"))]

def predict_gravity_system(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    combined = pd.concat([train.assign(_is_train=True), test.assign(_is_train=False)], ignore_index=True)
    combined = prepare(combined)
    combined["age_filled"] = fill_age(combined.loc[combined["_is_train"]].copy(), combined)
    train_mask = combined["_is_train"]
    y = combined.loc[train_mask, "survived"].astype(float)
    base = float(y.mean())
    eps = 1e-3
    base_logit = np.log((base + eps) / (1 - base + eps))
    score = pd.Series(0.0, index=combined.index)
    for _, rule_mask in gravity_rules(combined):
        support = int(rule_mask[train_mask].sum())
        if support < 5:
            continue
        rate = float(combined.loc[train_mask & rule_mask, "survived"].mean())
        weight = np.log((rate + eps) / (1 - rate + eps)) - base_logit
        weight *= min(1.0, support / 40.0)
        score.loc[rule_mask] += weight
    pred = (score >= 0).astype(int)
    test_rows = ~combined["_is_train"]
    return pd.DataFrame({"PassengerId": combined.loc[test_rows, "passengerid"].astype(int), "Survived": pred.loc[test_rows].astype(int)})


def oof_accuracy_ml(train: pd.DataFrame) -> float:
    y = train["Survived"].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in skf.split(train, y):
        fold_train = train.iloc[train_idx].copy()
        fold_valid = train.iloc[valid_idx].copy()
        model = build_model()
        model.fit(engineered_frame(fold_train, fold_train), fold_train["Survived"].astype(int))
        pred = model.predict(engineered_frame(fold_train, fold_valid)).astype(int)
        scores.append((pred == y[valid_idx]).mean())
    return float(sum(scores) / len(scores))

def oof_accuracy_group_table(train: pd.DataFrame) -> float:
    y = train["Survived"].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in skf.split(train, y):
        fold_train = train.iloc[train_idx].copy()
        fold_valid = train.iloc[valid_idx].drop(columns="Survived").copy()
        pred = predict_group_table(fold_train, fold_valid)
        pred = pred.set_index("PassengerId").loc[train.iloc[valid_idx]["PassengerId"]]["Survived"].astype(int).to_numpy()
        scores.append((pred == y[valid_idx]).mean())
    return float(sum(scores) / len(scores))

def oof_accuracy_male_third_class(train: pd.DataFrame) -> float:
    y = train["Survived"].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in skf.split(train, y):
        fold_train = train.iloc[train_idx].copy()
        fold_valid = train.iloc[valid_idx].drop(columns="Survived").copy()
        pred = predict_male_third_class(fold_train, fold_valid)
        pred = pred.set_index("PassengerId").loc[train.iloc[valid_idx]["PassengerId"]]["Survived"].astype(int).to_numpy()
        scores.append((pred == y[valid_idx]).mean())
    return float(sum(scores) / len(scores))

def oof_accuracy_death_rule(train: pd.DataFrame) -> float:
    y = train["Survived"].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in skf.split(train, y):
        fold_train = train.iloc[train_idx].copy()
        fold_valid = train.iloc[valid_idx].drop(columns="Survived").copy()
        pred = predict_death_rule(fold_train, fold_valid)
        pred = pred.set_index("PassengerId").loc[train.iloc[valid_idx]["PassengerId"]]["Survived"].astype(int).to_numpy()
        scores.append((pred == y[valid_idx]).mean())
    return float(sum(scores) / len(scores))

def oof_accuracy_dynamics_death(train: pd.DataFrame) -> float:
    y = train["Survived"].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in skf.split(train, y):
        fold_train = train.iloc[train_idx].copy()
        fold_valid = train.iloc[valid_idx].drop(columns="Survived").copy()
        pred = predict_dynamics_death(fold_train, fold_valid)
        pred = pred.set_index("PassengerId").loc[train.iloc[valid_idx]["PassengerId"]]["Survived"].astype(int).to_numpy()
        scores.append((pred == y[valid_idx]).mean())
    return float(sum(scores) / len(scores))

def oof_accuracy_gravity_system(train: pd.DataFrame) -> float:
    y = train["Survived"].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in skf.split(train, y):
        fold_train = train.iloc[train_idx].copy()
        fold_valid = train.iloc[valid_idx].drop(columns="Survived").copy()
        pred = predict_gravity_system(fold_train, fold_valid)
        pred = pred.set_index("PassengerId").loc[train.iloc[valid_idx]["PassengerId"]]["Survived"].astype(int).to_numpy()
        scores.append((pred == y[valid_idx]).mean())
    return float(sum(scores) / len(scores))

def oof_accuracy(train: pd.DataFrame, child_by_age: bool) -> float:
    y = train["Survived"].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in skf.split(train, y):
        fold_train = train.iloc[train_idx].copy()
        fold_valid = train.iloc[valid_idx].drop(columns="Survived").copy()
        pred = predict_rule(fold_train, fold_valid, child_by_age)
        pred = pred.set_index("PassengerId").loc[train.iloc[valid_idx]["PassengerId"]]["Survived"].astype(int).to_numpy()
        scores.append((pred == y[valid_idx]).mean())
    return float(sum(scores) / len(scores))
def score_frame(candidates: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(
        [
            {
                "file_name": row["file_name"],
                "model_name": row["model_name"],
                "predicted_survivors": row["predicted_survivors"],
                "predicted_non_survivors": row["predicted_non_survivors"],
                "oof_accuracy": row["oof_accuracy"],
                "validation_status": "ok",
            }
            for row in candidates
        ]
    ).sort_values(["oof_accuracy", "model_name"], ascending=[False, True], ignore_index=True)
    if frame.empty:
        return frame

    top = float(frame.loc[0, "oof_accuracy"])
    floor = float(frame["oof_accuracy"].min())
    spread = max(top - floor, 1e-9)
    frame["gravity"] = (frame["oof_accuracy"] - floor).round(6)
    frame["pull"] = ((frame["oof_accuracy"] - floor) / spread).round(6)
    frame["bucket"] = "candidate"
    frame.loc[0, "bucket"] = "primary"
    frame.loc[(frame["oof_accuracy"] > top - 0.03) & (frame["oof_accuracy"] < top - 0.01), "bucket"] = "candidate"
    frame.loc[frame["oof_accuracy"] >= top - 0.01, "bucket"] = "merge"
    frame.loc[0, "bucket"] = "primary"
    survival_rate = frame["predicted_survivors"] / 418.0
    frame.loc[survival_rate <= 0.38, "bucket"] = "negative_candidate"
    frame.loc[frame["model_name"].str.contains("death_rule|negative_rule|group_table_embarked"), "bucket"] = "negative_candidate"
    return frame

def merge_top_two(candidates: list[dict]) -> dict | None:
    if len(candidates) < 2:
        return None
    ranked = sorted(candidates, key=lambda row: (-row["oof_accuracy"], row["model_name"]))
    first, second = ranked[0], ranked[1]
    if first["oof_accuracy"] - second["oof_accuracy"] > 0.01:
        return None
    w1, w2 = first["oof_accuracy"], second["oof_accuracy"]
    merged = ((w1 * first["submission"]["Survived"] + w2 * second["submission"]["Survived"]) >= ((w1 + w2) / 2)).astype(int)
    merged_submission = first["submission"].copy()
    merged_submission["Survived"] = merged
    return {"file_name": OUT_PATH.name, "model_name": f"merge_{first['model_name']}_plus_{second['model_name']}", "predicted_survivors": int(merged_submission["Survived"].sum()), "predicted_non_survivors": int((1 - merged_submission["Survived"]).sum()), "oof_accuracy": float((w1 * first["oof_accuracy"] + w2 * second["oof_accuracy"]) / (w1 + w2)), "submission": merged_submission}

def vote_submission(candidates: list[dict], test: pd.DataFrame) -> pd.DataFrame:
    ranked = sorted(candidates, key=lambda row: (-row["oof_accuracy"], row["model_name"]))
    pool = ranked[:5]
    weights = np.array([max(float(row["oof_accuracy"]) - 0.5, 1e-3) for row in pool], dtype=float)
    votes = np.zeros(len(test), dtype=float)
    for weight, row in zip(weights, pool, strict=False):
        votes += weight * row["submission"]["Survived"].astype(int).to_numpy()
    final = (votes >= (weights.sum() / 2)).astype(int)
    return pd.DataFrame({"PassengerId": test["PassengerId"].astype(int), "Survived": final})

def validate(submission: pd.DataFrame, test: pd.DataFrame) -> None:
    if len(submission) != 418:
        fail("submission.csv must contain exactly 418 prediction rows.")
    if list(submission.columns) != ["PassengerId", "Survived"]:
        fail("submission.csv must contain exactly PassengerId and Survived columns.")
    if not set(submission["Survived"].unique()).issubset({0, 1}):
        fail("submission.csv Survived values must be only 0 or 1.")
    if not submission["PassengerId"].reset_index(drop=True).equals(test["PassengerId"].reset_index(drop=True)):
        fail("submission.csv PassengerId order must exactly match test.csv.")
    if sum(1 for _ in OUT_PATH.open("r", encoding="utf-8")) != 419:
        fail("submission.csv must contain 419 lines including the header.")

print("helpers loaded")


In [ ]:
# Edit this cell to try your own model.
# Return a DataFrame with columns: PassengerId, Survived.
def custom_candidate(train: pd.DataFrame, test: pd.DataFrame):
    return None


In [ ]:
candidates = []

candidate_specs = [
    ("woman_child_master_rule", lambda tr, te: predict_rule(tr, te, False), lambda tr: oof_accuracy(tr, False)),
    ("woman_child_age_rule", lambda tr, te: predict_rule(tr, te, True), lambda tr: oof_accuracy(tr, True)),
    ("logistic_embarked_ml", predict_ml, oof_accuracy_ml),
    ("group_table_embarked", predict_group_table, oof_accuracy_group_table),
    ("male_third_class_negative_rule", predict_male_third_class, oof_accuracy_male_third_class),
    ("male_death_rule", predict_death_rule, oof_accuracy_death_rule),
    ("dynamics_death_rule", predict_dynamics_death, oof_accuracy_dynamics_death),
    ("gravity_rule_system", predict_gravity_system, oof_accuracy_gravity_system),
]

for name, builder, scorer in candidate_specs:
    submission = builder(train, test)
    candidates.append(
        {
            "file_name": "submission.csv",
            "model_name": name,
            "predicted_survivors": int(submission["Survived"].sum()),
            "predicted_non_survivors": int((1 - submission["Survived"]).sum()),
            "oof_accuracy": float(scorer(train)),
            "submission": submission,
        }
    )

if custom_candidate is not None:
    custom_submission = custom_candidate(train, test)
    if custom_submission is not None:
        candidates.append(
            {
                "file_name": "submission.csv",
                "model_name": "custom_candidate",
                "predicted_survivors": int(custom_submission["Survived"].sum()),
                "predicted_non_survivors": int((1 - custom_submission["Survived"]).sum()),
                "oof_accuracy": float("nan"),
                "submission": custom_submission,
            }
        )

merge_candidate = merge_top_two(candidates)
if merge_candidate is not None:
    candidates.append(merge_candidate)

summary = score_frame(candidates)
submission = vote_submission(candidates, test)
submission.to_csv(uploads / "submission.csv", index=False)
validate(submission, test)

print(summary.to_string(index=False))
print("\nSaved:", uploads / "submission.csv")


In [ ]:
submission = pd.read_csv(uploads / "submission.csv")
print(submission.head().to_string(index=False))
print("rows:", len(submission))
